# Statystyka dla analityka danych — od podstaw do zaawansowanych

**Zakres:** miary tendencji centralnej, miary rozproszenia, kształt rozkładu (skośność/kurtoza), rozkłady prawdopodobieństwa, korelacje. Każda sekcja: co to jest, kiedy stosować, kiedy NIE stosować, przykład liczbowy + wykres z interpretacją.

**Filozofia doboru statystyki:** prawie każda para miar w tej notatce (średnia/mediana, wariancja/MAD, Pearson/Spearman) to wybór między **czułością** (lepiej wykorzystuje informację, ale wrażliwa na outliery/założenia) a **odpornością** (traci trochę informacji, ale stabilna niezależnie od jakości danych). Dobór zależy od tego, czy Twoim danym ufasz.

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

rng = np.random.default_rng(7)

# Dochód - naturalnie prawoskośny, z kilkoma wysokimi wartościami
income = rng.lognormal(mean=8.5, sigma=0.5, size=300).round(0)
income = np.append(income, [80000, 95000])

# Liczba zakupów - dyskretna, w miarę symetryczna
purchases = rng.poisson(3, 302)

df = pd.DataFrame({"income": income, "purchases": purchases})
df.describe()

# Część A — Miary tendencji centralnej

## A.1 — Średnia, mediana, dominanta

- **Średnia** — suma / liczba obserwacji. Wykorzystuje KAŻDĄ wartość, ale przez to jest wrażliwa na outliery.
- **Mediana** — wartość środkowa po posortowaniu. Odporna na outliery — zależy tylko od POZYCJI, nie wielkości ekstremalnych wartości.
- **Dominanta (mode)** — najczęściej występująca wartość. Jedyna miara sensowna dla danych KATEGORYCZNYCH; na danych CIĄGŁYCH zwykle bezużyteczna (patrz A.2).

In [ ]:
print(f"Średnia income:   {df['income'].mean():.0f}")
print(f"Mediana income:   {df['income'].median():.0f}")
print("-> średnia jest WYŻSZA niż mediana - klasyczny sygnał prawoskośnego rozkładu\n",
      "  (kilka wysokich wartości 'ciągnie' średnią w górę, mediana ich nie zauważa)")

### Wizualizacja: gdzie leżą względem siebie na rozkładzie skośnym

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(df["income"], bins=40, alpha=0.7)
ax.axvline(df["income"].mean(), color="red", linestyle="--", label=f"Średnia = {df['income'].mean():.0f}")
ax.axvline(df["income"].median(), color="green", linestyle="--", label=f"Mediana = {df['income'].median():.0f}")
ax.set_title("Income: średnia vs mediana na rozkładzie prawoskośnym")
ax.legend()

**Interpretacja:** średnia (czerwona) leży wyraźnie na PRAWO od mediany (zielona) — obie "widzą" inny środek danych. Przy rozkładzie skośnym mediana lepiej reprezentuje "typową" obserwację; średnia lepiej reprezentuje sumę/budżet (bo z definicji `suma = średnia × n`). **Kiedy używać której:** mediana do komunikowania "typowego klienta/pracownika"; średnia, gdy interesuje Cię wartość zagregowana (np. do przeliczenia na budżet całkowity).

## A.2 — Dominanta: kiedy ma sens, a kiedy nie

Na danych DYSKRETNYCH (liczba zakupów, kategoria, ocena 1–5) dominanta jest naturalna i czytelna. Na danych CIĄGŁYCH (dochód, czas, waga) niemal każda wartość jest unikalna — "najczęstsza wartość" bywa przypadkowa i niereprezentatywna.

In [ ]:
print("Dominanta na danych DYSKRETNYCH (liczba zakupów) - ma sens:")
print(f"  Mode: {df['purchases'].mode().tolist()}")
print(df["purchases"].value_counts().sort_index().head(6))
print()
print("Dominanta na danych CIĄGŁYCH (income) - bezużyteczna:")
print(f"  Mode: {df['income'].mode().tolist()[:5]} (przypadkowe remisy, nic nie mówią o rozkładzie)")

## A.3 — Średnia ucinana (trimmed mean)

Kompromis między średnią (czuła, wykorzystuje wszystko) a medianą (odporna, ale ignoruje kształt reszty danych): odetnij ustalony % obserwacji z KAŻDEGO końca posortowanych danych, policz średnią z reszty. Im większy procent odcięcia, tym bliżej mediany.

In [ ]:
for cut in [0.0, 0.05, 0.1, 0.2]:
    tm = stats.trim_mean(df["income"], proportiontocut=cut)
    print(f"trim_mean(cut={cut:.2f}): {tm:.0f}")
print(f"\nDla porównania - mediana: {df['income'].median():.0f}")
print("Kiedy stosować: raporty odporne na pojedyncze błędy danych, ale gdzie zależy Ci",
      "\nna wykorzystaniu WIĘKSZOŚCI informacji (nie tylko pozycji środkowej jak mediana).",
      "\nTypowe w sporcie (ocena sędziowska - odrzuca się najwyższą/najniższą notę).")

## A.4 — Średnia ważona

Zwykła średnia traktuje każdą obserwację jednakowo. Średnia ważona nadaje różnym obserwacjom różną "siłę głosu" — naturalne, gdy niektóre pomiary są bardziej wiarygodne/liczne niż inne (np. średnia ocena kilku sklepów, gdzie jeden ma 800 opinii, a inny tylko 15).

In [ ]:
store_scores = pd.DataFrame({
    "store": ["Warszawa", "Krakow", "Gdansk", "Poznan"],
    "avg_rating": [4.2, 4.8, 4.5, 3.9],
    "n_reviews": [500, 15, 200, 800],
})

simple_mean = store_scores["avg_rating"].mean()
weighted_mean = np.average(store_scores["avg_rating"], weights=store_scores["n_reviews"])

print(store_scores)
print(f"\nZwykła średnia ocen sklepów:        {simple_mean:.2f}")
print(f"Średnia ważona liczbą opinii:        {weighted_mean:.2f}")
print("\n-> Kraków ma najwyższą pojedynczą ocenę (4.8), ale tylko 15 opinii,",
      "\n   więc słabo wpływa na wynik ważony liczbą recenzji - zgodnie z intuicją,",
      "\n   że 15 opinii niesie mniej wiarygodnej informacji niż 800.")

## A.5 — Średnia bayesowska (bayesian average / shrinkage estimator)

Rozwiązuje konkretny problem średniej ważonej: co zrobić z produktem, który ma ŚWIETNĄ ocenę, ale z bardzo MAŁEJ liczby opinii (np. 5.0 z 2 recenzji)? Naiwnie taki produkt wygrałby ranking z produktem mającym 4.7 z 800 recenzji — mimo że ta druga ocena jest znacznie bardziej wiarygodna.

**Mechanizm:** "dokładamy" `m` wirtualnych opinii o wartości równej globalnej średniej `C` (tzw. prior) do każdego produktu, zanim policzymy średnią. Im MNIEJ prawdziwych opinii ma produkt, tym mocniej wynik jest "ściągany" w stronę globalnej średniej.

$$\text{bayesian\_avg} = \frac{v}{v+m} \cdot R + \frac{m}{v+m} \cdot C$$

gdzie `v` = liczba opinii produktu, `R` = jego średnia ocena, `C` = globalna średnia (prior), `m` = próg "wiarygodności" (im większe `m`, tym mocniejsze ściąganie w stronę `C`).

In [ ]:
products = pd.DataFrame({
    "product": ["A", "B", "C", "D"],
    "avg_rating": [5.0, 4.6, 4.9, 4.7],
    "n_reviews": [2, 5000, 3, 800],
})

C = 4.0  # globalna średnia ocen w całym serwisie (prior - NIE liczona tylko z tych 4 produktów)
m = 50   # próg wiarygodności: ile 'wirtualnych' opinii o wartości C dokładamy każdemu produktowi

products["bayesian_avg"] = (
    (products["n_reviews"] / (products["n_reviews"] + m)) * products["avg_rating"]
    + (m / (products["n_reviews"] + m)) * C
)

print(products.round(3))
print()
print("Ranking naiwny (wg avg_rating):    ", products.sort_values("avg_rating", ascending=False)["product"].tolist())
print("Ranking bayesowski (wg bayesian_avg):", products.sort_values("bayesian_avg", ascending=False)["product"].tolist())

**Interpretacja:** ranking naiwny stawia produkty `A` i `C` (2–3 opinie!) na szczycie tylko dlatego, że przez przypadek dostały same wysokie oceny. Ranking bayesowski poprawnie przesuwa na górę `D` i `B` — mają dużo niższe pojedyncze oceny, ale poparte tysiącami recenzji, więc bardziej wiarygodne. **Zastosowanie:** rankingi produktów/sprzedawców/recenzji, gdzie liczba obserwacji mocno się różni między pozycjami — dokładnie ten mechanizm stoi za rankingami IMDb czy algorytmami rekomendacji "najlepiej ocenianych" pozycji.

**Kiedy NIE stosować:** gdy liczba obserwacji jest podobna dla wszystkich porównywanych elementów — wtedy shrinkage nie zmienia rankingu, tylko komplikuje obliczenia bez potrzeby. Wybór `m` jest subiektywny — warto go dobrać tak, by odpowiadał "rozsądnej" liczbie opinii, od której ufasz średniej.

# Część B — Miary rozproszenia

## B.1 — Rozstęp, wariancja, odchylenie standardowe

- **Rozstęp (range)** = max − min. Prosty, ale zależy WYŁĄCZNIE od dwóch skrajnych wartości — jeden outlier go dominuje.
- **Wariancja** — średni kwadrat odchylenia od średniej. Jednostka to KWADRAT jednostki oryginalnej (np. PLN²) — trudna do interpretacji wprost.
- **Odchylenie standardowe** — pierwiastek z wariancji, z powrotem w oryginalnych jednostkach. Najczęściej używana miara rozproszenia.

**Przypomnienie o `ddof`** (już poruszone w notatce o skalowaniu cech): `pandas`/`polars` domyślnie liczą wariancję/std dzieląc przez `n-1` (`ddof=1`, wariancja PRÓBY), `numpy` domyślnie przez `n` (`ddof=0`, wariancja POPULACJI).

In [ ]:
print(f"Rozstęp: {df['income'].max() - df['income'].min():.0f}")
print(f"Wariancja (pandas, ddof=1): {df['income'].var():.0f}")
print(f"Odch. std (pandas, ddof=1): {df['income'].std():.2f}")
print(f"Odch. std (numpy, ddof=0):  {df['income'].to_numpy().std():.2f}")

## B.2 — IQR (rozstęp międzykwartylowy)

`Q3 − Q1` — rozpiętość ŚRODKOWYCH 50% danych. Ignoruje skrajne 25% z każdej strony, więc jest naturalnie odporny na outliery (podstawa "wąsów" na boxplocie i metody wykrywania outlierów z poprzedniej notatki).

In [ ]:
q1, q3 = df["income"].quantile([0.25, 0.75])
print(f"Q1={q1:.0f}, Q3={q3:.0f}, IQR={q3-q1:.0f}")

fig, ax = plt.subplots(figsize=(7, 3))
sns.boxplot(x=df["income"], ax=ax)
ax.set_title("Boxplot income - pudełko to właśnie IQR (Q1 do Q3)")

## B.3 — MAD: UWAGA na DWA różne skróty pod tą samą nazwą

"MAD" w statystyce oznacza DWIE różne rzeczy, w zależności od źródła:
- **Mean Absolute Deviation** — średnia z `|x - średnia|`.
- **Median Absolute Deviation** — mediana z `|x - mediana|` (to ten "MAD", którego użyliśmy w notatce o outlierach do zmodyfikowanego Z-score — znacznie bardziej odporny, bo oparty na medianie w OBU krokach).

In [ ]:
mean_abs_dev = (df["income"] - df["income"].mean()).abs().mean()
median_abs_dev = stats.median_abs_deviation(df["income"])
median_abs_dev_scaled = stats.median_abs_deviation(df["income"], scale="normal")

print(f"Mean Absolute Deviation (od średniej):        {mean_abs_dev:.2f}")
print(f"Median Absolute Deviation (od mediany):        {median_abs_dev:.2f}")
print(f"Median Absolute Deviation, scale='normal':     {median_abs_dev_scaled:.2f}")
print(f"Dla porównania, odch. std:                     {df['income'].std():.2f}")
print("\n`scale='normal'` przelicza medianę MAD tak, by była w przybliżeniu",
      "\nporównywalna z odchyleniem std NA DANYCH NORMALNYCH - dzięki temu można",
      "\nużywać jej zamiennie z std, ale z odpornością na outliery.")

## B.4 — Współczynnik zmienności (CV)

`CV = odch. std / średnia` — pozwala porównać ROZPROSZENIE zmiennych o zupełnie różnych jednostkach/skalach (np. "czy dochód jest bardziej zróżnicowany niż wiek?" — bezpośrednie porównanie std nie ma sensu, bo jednostki są nieporównywalne).

In [ ]:
age_demo = rng.normal(40, 10, 300)

cv_income = df["income"].std() / df["income"].mean()
cv_age = age_demo.std() / age_demo.mean()

print(f"CV income: {cv_income:.2f}  (std stanowi {cv_income*100:.0f}% średniej)")
print(f"CV age:    {cv_age:.2f}  (std stanowi {cv_age*100:.0f}% średniej)")
print("-> mimo że std(income) jest liczbowo dużo większe niż std(age),",
      "\n   dopiero CV pokazuje, która zmienna jest WZGLĘDNIE bardziej zróżnicowana")

# Część C — Kształt rozkładu

## C.1 — Skośność (skewness)

Mierzy ASYMETRIĘ rozkładu. `0` = symetryczny, `>0` = ogon w prawo (typowe dla dochodów/cen), `<0` = ogon w lewo (rzadsze — np. wiek śmierci w populacji o dobrej opiece zdrowotnej).

In [ ]:
symmetric = rng.normal(0, 1, 5000)
right_skewed = rng.lognormal(0, 1, 5000)
left_skewed = -rng.lognormal(0, 1, 5000)

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
for ax, data, title in zip(
    axes,
    [symmetric, right_skewed, left_skewed],
    ["Symetryczny", "Prawoskośny", "Lewoskośny"],
):
    ax.hist(data, bins=40)
    ax.set_title(f"{title}\nskośność={stats.skew(data):.2f}")
plt.tight_layout()

**Interpretacja praktyczna:** `|skośność| < 0.5` — w przybliżeniu symetryczny, średnia/mediana blisko siebie. `0.5–1` — umiarkowana skośność, warto sprawdzić medianę obok średniej. `>1` — silna skośność, mediana zdecydowanie lepiej reprezentuje "typową" wartość niż średnia.

## C.2 — Kurtoza (kurtosis)

Mierzy "grubość ogonów" — jak często zdarzają się wartości EKSTREMALNE, względem rozkładu normalnego. **Uwaga:** `scipy.stats.kurtosis()` domyślnie zwraca **excess kurtosis** (kurtoza − 3), więc rozkład normalny wychodzi ~0, nie ~3 (klasyczna kurtoza Pearsona — dostępna przez `fisher=False`).

In [ ]:
normal_data = rng.normal(0, 1, 5000)
uniform_data = rng.uniform(-1.73, 1.73, 5000)
t_data = rng.standard_t(df=3, size=5000)

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5), sharex=True)
for ax, data, title in zip(
    axes,
    [uniform_data, normal_data, t_data],
    ["Jednostajny (platykurtyczny)", "Normalny (mezokurtyczny)", "t, df=3 (leptokurtyczny)"],
):
    ax.hist(data, bins=50, range=(-6, 6), density=True)
    ax.set_title(f"{title}\nexcess kurtoza={stats.kurtosis(data):.2f}")
plt.tight_layout()

**Interpretacja praktyczna:** wysoka dodatnia kurtoza (leptokurtyczny) oznacza więcej ekstremalnych wartości niż "spodziewa się" rozkład normalny — realne ryzyko niedoszacowania rzadkich, dużych zdarzeń, jeśli model zakłada normalność (klasyczny problem w finansach — "czarne łabędzie").

## C.3 — Percentyle / kwantyle

Percentyl `p` to wartość, poniżej której leży `p%` obserwacji. Mediana to percentyl 50. Odporne na outliery (jak IQR) — patrzą na POZYCJĘ, nie wielkość skrajnych wartości.

In [ ]:
percentiles = df["income"].quantile([0.05, 0.25, 0.5, 0.75, 0.95, 0.99])
percentiles

# Część D — Rozkłady prawdopodobieństwa

## D.1 — Rozkład normalny (Gaussa)

Najważniejszy rozkład ciągły — symetryczny, w pełni opisany przez średnią i odchylenie std. **Reguła 68-95-99.7**: w obrębie 1/2/3 odchyleń std od średniej mieści się odpowiednio ~68%/~95%/~99.7% obserwacji.

In [ ]:
normal_sample = rng.normal(100, 15, 100_000)
mean, std = normal_sample.mean(), normal_sample.std()

for k in [1, 2, 3]:
    within = ((normal_sample > mean - k * std) & (normal_sample < mean + k * std)).mean()
    print(f"W obrębie {k} odch. std: {within*100:.1f}%")

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(normal_sample, bins=60, density=True, alpha=0.6)
x = np.linspace(normal_sample.min(), normal_sample.max(), 200)
ax.plot(x, stats.norm.pdf(x, mean, std), "r-", linewidth=2)
for k, color in zip([1, 2, 3], ["green", "orange", "red"]):
    ax.axvline(mean + k * std, color=color, linestyle="--", alpha=0.6)
    ax.axvline(mean - k * std, color=color, linestyle="--", alpha=0.6)
ax.set_title("Rozkład normalny z zaznaczonymi 1/2/3 odch. std")

## D.2 — Pozostałe kluczowe rozkłady: przegląd wizualny

- **Jednostajny** — każda wartość w zakresie równie prawdopodobna (np. czas oczekiwania w ustalonym oknie).
- **Dwumianowy** — liczba sukcesów w `n` niezależnych próbach binarnych (np. liczba konwersji na 100 wejść).
- **Poissona** — liczba zdarzeń w ustalonym czasie/przestrzeni (np. liczba zgłoszeń serwisowych dziennie).
- **Wykładniczy** — czas DO następnego zdarzenia (np. czas między kolejnymi zamówieniami).
- **Log-normalny** — zmienna, której LOGARYTM ma rozkład normalny (typowe dla dochodów, cen, czasu trwania sesji).

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(13, 7))

axes[0, 0].hist(rng.uniform(0, 10, 5000), bins=30)
axes[0, 0].set_title("Jednostajny [0,10]")

axes[0, 1].hist(rng.binomial(100, 0.3, 5000), bins=30)
axes[0, 1].set_title("Dwumianowy (n=100, p=0.3)")

axes[0, 2].hist(rng.poisson(4, 5000), bins=range(0, 15))
axes[0, 2].set_title("Poissona (λ=4)")

axes[1, 0].hist(rng.exponential(scale=5, size=5000), bins=40)
axes[1, 0].set_title("Wykładniczy (scale=5)")

axes[1, 1].hist(rng.lognormal(0, 0.5, 5000), bins=40)
axes[1, 1].set_title("Log-normalny")

axes[1, 2].hist(rng.normal(0, 1, 5000), bins=40)
axes[1, 2].set_title("Normalny (dla porównania)")

plt.tight_layout()

## D.3 — Testowanie normalności: Shapiro-Wilk i QQ-plot

Shapiro-Wilk daje formalną odpowiedź liczbową (p-value); QQ-plot daje odpowiedź WIZUALNĄ i pokazuje, W KTÓRYM miejscu rozkład odbiega od normalności (ogony? środek?) — informacja, której sam p-value nie daje.

In [ ]:
sample_normal = rng.normal(0, 1, 200)
sample_skewed = rng.lognormal(0, 1, 200)

stat_n, p_n = stats.shapiro(sample_normal)
stat_s, p_s = stats.shapiro(sample_skewed)
print(f"Shapiro-Wilk, dane normalne: p-value={p_n:.4f} (p>0.05 -> brak podstaw do odrzucenia normalności)")
print(f"Shapiro-Wilk, dane skośne:   p-value={p_s:.6f} (p<0.05 -> odrzucamy normalność)")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
stats.probplot(sample_normal, dist="norm", plot=axes[0])
axes[0].set_title("QQ-plot: dane normalne\n(punkty na linii = zgodność z rozkładem normalnym)")
stats.probplot(sample_skewed, dist="norm", plot=axes[1])
axes[1].set_title("QQ-plot: dane skośne\n(odchylenie od linii w ogonach = brak normalności)")
plt.tight_layout()

## D.4 — Dopasowanie rozkładu do danych: `.fit()`

Gdy podejrzewasz konkretny typ rozkładu (np. czas między zdarzeniami → wykładniczy), `scipy.stats` potrafi dopasować parametry metodą największej wiarygodności.

In [ ]:
data_for_fit = rng.exponential(scale=5, size=1000)
loc, scale = stats.expon.fit(data_for_fit)
print(f"Dopasowane parametry: loc={loc:.2f}, scale={scale:.2f} (prawdziwy scale=5)")

# Część E — Korelacje

## E.1 — Pearson vs Spearman: zależność LINIOWA vs MONOTONICZNA

- **Pearson** — mierzy siłę zależności LINIOWEJ. Zakłada w miarę liniowy związek.
- **Spearman** — mierzy zależność MONOTONICZNĄ (rosnąco lub malejąco, niekoniecznie liniowo) — liczony na RANGACH, nie surowych wartościach.

In [ ]:
x = rng.uniform(1, 10, 100)
y = x**3  # silna zależność, ale NIEliniowa

pearson_r, _ = stats.pearsonr(x, y)
spearman_r, _ = stats.spearmanr(x, y)

fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(x, y, alpha=0.5)
ax.set_title(f"y = x³   |   Pearson={pearson_r:.3f}, Spearman={spearman_r:.3f}")

print("Spearman = 1.0: idealnie wykrywa PEŁNĄ zależność monotoniczną.")
print("Pearson niedoszacowuje: zależność jest silna, ale nie jest linią prostą.")

## E.2 — Wpływ POJEDYNCZEGO outliera: Pearson vs Spearman

In [ ]:
x2 = rng.normal(0, 1, 100)
y2 = x2 + rng.normal(0, 0.3, 100)

pearson_clean, _ = stats.pearsonr(x2, y2)
spearman_clean, _ = stats.spearmanr(x2, y2)

x2_out, y2_out = x2.copy(), y2.copy()
x2_out[0], y2_out[0] = 50, -50  # jeden skrajny, psujący outlier

pearson_out, _ = stats.pearsonr(x2_out, y2_out)
spearman_out, _ = stats.spearmanr(x2_out, y2_out)

print(f"Pearson  bez outliera: {pearson_clean:+.3f}   |   z outlierem: {pearson_out:+.3f}")
print(f"Spearman bez outliera: {spearman_clean:+.3f}   |   z outlierem: {spearman_out:+.3f}")
print("\n-> JEDEN outlier ODWRÓCIŁ ZNAK korelacji Pearsona (z dodatniej na ujemną!),")
print("   Spearman ledwo drgnął - bo opiera się na RANGACH, nie surowych wartościach")

## E.3 — Kendall's tau

Kolejna korelacja rankingowa (jak Spearman), ale liczona inaczej — na podstawie par zgodnych/niezgodnych rankingów. Zalecana przy MAŁYCH próbach i danych z wieloma remisami (rankingami o tej samej wartości), gdzie bywa stabilniejsza niż Spearman.

In [ ]:
kendall_tau, _ = stats.kendalltau(x, y)
print(f"Kendall tau: {kendall_tau:.3f}")

## E.4 — Macierz korelacji jako heatmapa

In [ ]:
multi_df = pd.DataFrame({
    "income": df["income"],
    "purchases": df["purchases"],
    "random_noise": rng.normal(0, 1, len(df)),
})

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(multi_df.corr(method="spearman"), annot=True, cmap="coolwarm", center=0, vmin=-1, vmax=1, ax=ax)
ax.set_title("Macierz korelacji Spearmana")

# Pułapki

### Pułapka 1 — korelacja ≠ przyczynowość, nawet przy bardzo wysokim współczynniku

Wysoka korelacja (nawet 0.99) mówi tylko, że dwie zmienne poruszają się razem — nie mówi, która wpływa na którą, ani czy w ogóle istnieje bezpośredni związek (mogą mieć wspólną przyczynę zewnętrzną). Żadna z metod w Części E tego nie rozstrzyga.

### Pułapka 2 — dominanta na danych ciągłych to często artefakt zaokrąglenia, nie sygnał

Jak pokazano w A.2 — na surowych danych ciągłych "najczęstsza wartość" bywa czystym przypadkiem (kilka identycznych odczytów z powodu zaokrąglenia pomiaru), nie realną cechą rozkładu.

### Pułapka 3 — dwa różne "MAD" pod tą samą nazwą (B.3)

Mylenie Mean Absolute Deviation z Median Absolute Deviation to częsty błąd komunikacyjny między analitykami — zawsze warto doprecyzować, o który "MAD" chodzi, szczególnie w dokumentacji/raportach.

### Pułapka 4 — p-value z testu normalności zależy od wielkości próby

Przy bardzo dużych próbach (dziesiątki tysięcy obserwacji) test Shapiro-Wilk potrafi odrzucić normalność nawet przy praktycznie nieistotnych, kosmetycznych odchyleniach — bo test staje się coraz czulszy wraz ze wzrostem `n`. Warto łączyć wynik testu z wizualną oceną (QQ-plot, histogram), nie polegać wyłącznie na p-value.

# Podsumowanie

| Sytuacja | Zalecana miara |
|---|---|
| Dane skośne, potrzebna "typowa" wartość | Mediana |
| Dane skośne, potrzebna wartość do zsumowania/budżetu | Średnia |
| Dane kategoryczne/dyskretne, najczęstsza kategoria | Dominanta |
| Kompromis odporność/wykorzystanie informacji | Średnia ucinana |
| Niektóre obserwacje bardziej wiarygodne/liczne niż inne | Średnia ważona |
| Ranking wg wskaźnika z bardzo różną liczbą obserwacji na pozycję | Średnia bayesowska (shrinkage) |
| Rozproszenie, dane bez outlierów | Odchylenie standardowe |
| Rozproszenie, dane z outlierami | IQR lub Median Absolute Deviation |
| Porównanie rozproszenia zmiennych o różnych jednostkach | Współczynnik zmienności (CV) |
| Czy rozkład symetryczny | Skośność |
| Czy rozkład ma "ciężkie ogony" (ryzyko ekstremów) | Kurtoza (excess) |
| Zależność liniowa, dane bez outlierów | Korelacja Pearsona |
| Zależność monotoniczna (niekoniecznie liniowa) lub dane z outlierami | Korelacja Spearmana |
| Mała próba, dużo remisów w rankingu | Korelacja Kendalla |
| Test formalny normalności | Shapiro-Wilk (+ zawsze QQ-plot obok) |

**Wniosek:** ten sam motyw co w notatkach o skalowaniu i outlierach — każda "czuła" statystyka (średnia, wariancja, Pearson) ma odporny odpowiednik (mediana, MAD/IQR, Spearman). Wybór między nimi to świadoma decyzja o tym, ile ufasz jakości swoich danych, nie kwestia "która jest poprawna".